# Beyond the Dot Product on a rented H100

Plunkett's experiment end to end on Qwen3, on a box that is deleted at expiry.

Run the cells in order. The smoke config proves the GPU path in a few minutes
before the long run starts, which is the cheapest way to find a broken setting.

**The machine is wiped at expiry and there is no recovery.** Push after every
stage, not at the end. The persist cells below send results and adapters to the
Hugging Face Hub; code changes go back to GitHub yourself.

In [ ]:
!nvidia-smi

In [ ]:
# Clone the pipeline branch and install. run_me.py must run before the install:
# it re-pins requirements to whatever this box already ships, which keeps its
# CUDA-matched torch instead of letting pip swap it out.
import os, subprocess, sys

REPO = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", "pipeline", REPO], check=True)
    os.chdir("Decision_Task_Database_Experiments")
subprocess.run([sys.executable, "run_me.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Keep model downloads in one folder so they are easy to delete when done.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
print("ready in", os.getcwd())

In [ ]:
# Confirm the GPU is what the settings assume. bf16 needs Ampere or newer;
# the configs below ask for it, and the loader refuses rather than silently
# falling back.
import torch

print("torch", torch.__version__)
print("gpu", torch.cuda.get_device_name(0))
print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("bf16 supported", torch.cuda.is_bf16_supported())

In [ ]:
# Credentials. Typed, never written into the notebook, so nothing lands in git.
import getpass, os

os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face write token: ")
HUB_REPO = "CHANGE-ME/bdp-plunkett-qwen3"   # <- your Hub repo; created on first push
print("pushing runs to", HUB_REPO)

## 1. Smoke run

Qwen3-0.6B, sixty steps, ten personas. Proves the CUDA path, the estimator, the
report parser and the results writer. Not a result.

In [ ]:
from src.config import load
from src.pipeline import run

smoke = run(load("configs/h100_smoke.json"))

In [ ]:
from src.persist import push_run

push_run(smoke[-1]["run_id"], HUB_REPO)

## 2. The replication

Qwen3-8B, Plunkett's settings otherwise: 100 personas, 1500 steps, a checkpoint
every 300, then introspection training in two folds. The checkpoints are the
point, since they are what shows whether faithfulness arrives later than
decision accuracy.

Push again as soon as it finishes.

In [ ]:
eight = run(load("configs/h100_plunkett_8b.json"))

In [ ]:
push_run(eight[-1]["run_id"], HUB_REPO)

## 3. Stretch run

Only if the clock allows. Qwen3-14B in bf16 is about 28 GB of weights, which
fits the card and stays inside the box's disk guidance. 32B does not: its
weights alone are past the limit you were asked to respect.

In [ ]:
fourteen = run(load("configs/h100_plunkett_14b.json"))
push_run(fourteen[-1]["run_id"], HUB_REPO)

## 4. Before you walk away

Sweeps everything still on disk to the Hub, including any run interrupted
part way. The pipeline appends results as it goes, so a partial run is still
worth saving.

In [ ]:
from src.persist import run_ids, push_run

for rid in run_ids():
    push_run(rid, HUB_REPO)